# Practical 04: Out-of-Core Batch Ingestion & Aggregation via Chunking

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to ingest a large CSV file exceeding single-process RAM in batches using the chunksize parameter of pd.read_csv(). Calculate the cumulative record count, running sum of transaction amounts, and overall average transaction value across all chunks to prevent Out-Of-Memory (OOM) crashes.

In [1]:
import os
import shutil
import tempfile
import numpy as np
import pandas as pd

TOTAL_ROWS = 800_000
WRITE_BATCH = 160_000
CHUNK_SIZE = 80_000

rng = np.random.default_rng(7)
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "large_transactions.csv")

written = 0
while written < TOTAL_ROWS:
    m = min(WRITE_BATCH, TOTAL_ROWS - written)
    batch = pd.DataFrame({
        "transaction_id": np.arange(written + 1, written + m + 1),
        "customer_id": rng.integers(1, 45000, m),
        "amount": np.round(rng.gamma(2.0, 55.0, m), 2),
        "city": rng.choice(["Surat", "Delhi", "Pune", "Jaipur", "Indore"], m),
    })
    batch.to_csv(csv_path, mode="a", header=(written == 0), index=False)
    written += m

print(f"File created: {os.path.getsize(csv_path) / 1024**2:.1f} MB, {TOTAL_ROWS:,} rows")

total_records = 0
total_amount = 0.0

reader = pd.read_csv(csv_path, usecols=["transaction_id", "amount"], chunksize=CHUNK_SIZE)

print(f"{'Chunk':>5} | {'Rows':>8} | {'Cum. records':>12} | {'Running sum':>16} | {'Running avg':>11}")
print("-" * 66)
for i, chunk in enumerate(reader, start=1):
    total_records += len(chunk)
    total_amount += chunk["amount"].sum()
    print(f"{i:>5} | {len(chunk):>8,} | {total_records:>12,} | {total_amount:>16,.2f} | "
          f"{total_amount / total_records:>11.4f}")

average = total_amount / total_records
print(f"Total records             : {total_records:,}")
print(f"Total transaction amount  : {total_amount:,.2f}")
print(f"Average transaction value : {average:.4f}")

full = pd.read_csv(csv_path, usecols=["amount"])["amount"]
print(f"Full-load sum : {full.sum():,.2f} | mean : {full.mean():.4f}")

shutil.rmtree(tmp_dir)
print("Temporary files removed.")

File created: 19.2 MB, 800,000 rows
Chunk |     Rows | Cum. records |      Running sum | Running avg
------------------------------------------------------------------
    1 |   80,000 |       80,000 |     8,792,174.00 |    109.9022
    2 |   80,000 |      160,000 |    17,585,831.59 |    109.9114
    3 |   80,000 |      240,000 |    26,344,862.05 |    109.7703
    4 |   80,000 |      320,000 |    35,097,288.65 |    109.6790
    5 |   80,000 |      400,000 |    43,890,312.40 |    109.7258
    6 |   80,000 |      480,000 |    52,692,076.22 |    109.7752
    7 |   80,000 |      560,000 |    61,471,926.30 |    109.7713
    8 |   80,000 |      640,000 |    70,257,145.28 |    109.7768
    9 |   80,000 |      720,000 |    79,101,671.85 |    109.8634
   10 |   80,000 |      800,000 |    87,930,361.83 |    109.9130
Total records             : 800,000
Total transaction amount  : 87,930,361.83
Average transaction value : 109.9130
Full-load sum : 87,930,361.83 | mean : 109.9130
Temporary files rem